# Spec-FastGS — BOSCH Setup Environment & Build Wheels
Run once with the **JupyterHub default kernel** (Python 3.10/3.11).  
Creates the `thesis_env` virtual environment, compiles Spec-FastGS custom CUDA extensions (`diff-gaussian-rasterization_fastgs`, `simple-knn`, `fused-ssim`), and installs required libraries.

**What this installs:**
1. Custom PyTorch virtual environment (`thesis_env`) matching system CUDA.
2. Compiles and installs `diff-gaussian-rasterization_fastgs`, `simple-knn`, and `fused-ssim` submodules.
3. Registers the environment as `Python (thesis_env)` Jupyter kernel.

| Cell | Purpose |
|------|---------|
| `c00_proxy` | Set BOSCH proxy env vars |
| `c01_config` | Paths and constants |
| `c02_create_env` | Create `thesis_env` venv (python -m venv) |
| `c03_cuda_search` | Search system `nvcc` and load matching CUDA modules |
| `c04_pytorch` | Install PyTorch matching the active CUDA compiler |
| `c05_base_reqs` | Install base packages (`wheel`, `setuptools`, `ninja`, `numpy<2`, `huggingface_hub`, etc.) |
| `c06_cuda_ext` | Compile and install custom CUDA submodules as wheels |
| `c07_kernel` | Register `thesis_env` as a Jupyter kernel |
| `c08_verify` | Import verification check — all rows must PASS |

## c00 — Proxy
Sets the BOSCH proxy for all subsequent pip / git / HuggingFace downloads. Run this **first**.

In [ ]:
# ── Proxy (required for pip / git / HuggingFace downloads) ─────────────────────────
import os

PROXY = 'http://rb-proxy-sl.bosch.com:8080'
HOME  = os.path.expanduser('~')

os.environ['http_proxy']  = PROXY
os.environ['https_proxy'] = PROXY
os.environ['HTTP_PROXY']  = PROXY
os.environ['HTTPS_PROXY'] = PROXY

print(f'Proxy set: {PROXY}')

## c01 — Paths & Config
Defines path variables and verifies required repository directories.

In [ ]:
# ── Paths & constants ─────────────────────────────────────────────────────────
import os

HOME = os.path.expanduser('~')

# Robustly resolve repository root path
if os.path.isdir('/home/ghp4hc/thesis-all/spec-fastgs'):
    REPO_ROOT = '/home/ghp4hc/thesis-all/spec-fastgs'
elif os.path.isdir(os.path.join(os.getcwd(), 'thesis-all', 'spec-fastgs')):
    REPO_ROOT = os.path.join(os.getcwd(), 'thesis-all', 'spec-fastgs')
else:
    REPO_ROOT = os.path.join(os.getcwd(), 'spec-fastgs')

SUBMODULES_DIR = os.path.join(REPO_ROOT, 'submodules')
ENV_NAME       = 'thesis_env'

print(f'HOME          : {HOME}')
print(f'REPO_ROOT     : {REPO_ROOT}')
print(f'SUBMODULES_DIR: {SUBMODULES_DIR}')
print(f'ENV_NAME      : {ENV_NAME}')

assert os.path.isdir(REPO_ROOT), f'Repository not found at {REPO_ROOT}. Make sure the repo is cloned correctly.'
assert os.path.isdir(SUBMODULES_DIR), f'Submodules folder not found at {SUBMODULES_DIR}'

## c02 — Create venv
Creates `thesis_env` using `python -m venv`.

In [ ]:
# ── Create thesis_env venv ───────────────────────────────────────────────────
import subprocess
import json
import os
import sys
import shutil

PROXY    = 'http://rb-proxy-sl.bosch.com:8080'
ENV_NAME = 'thesis_env'

os.environ.update({'http_proxy': PROXY, 'https_proxy': PROXY,
                   'HTTP_PROXY': PROXY, 'HTTPS_PROXY': PROXY})

r_info   = subprocess.run(['conda', 'info', '--json'], capture_output=True, text=True)
if r_info.returncode == 0:
    info     = json.loads(r_info.stdout)
    envs_dir = info.get('envs_dirs', [f'{HOME}/.conda/envs'])[0]
else:
    envs_dir = f'{HOME}/.conda/envs'

thesis_env    = os.path.join(envs_dir, ENV_NAME)
thesis_python = os.path.join(thesis_env, 'bin', 'python')
thesis_pip    = os.path.join(thesis_env, 'bin', 'pip')

if os.path.isfile(thesis_python):
    print(f'Env already exists at {thesis_env}')
else:
    candidates = [
        shutil.which('python3.10'), '/opt/conda/bin/python3.10',
        shutil.which('python3.11'), '/opt/conda/bin/python3.11', sys.executable
    ]
    base_python = next((p for p in candidates if p and os.path.isfile(p)), sys.executable)
    print(f'Base Python : {base_python}')
    os.makedirs(envs_dir, exist_ok=True)
    r_venv = subprocess.run([base_python, '-m', 'venv', thesis_env], capture_output=True, text=True)
    if r_venv.returncode != 0:
        raise RuntimeError(f'venv creation failed: {r_venv.stderr}')
    subprocess.run([thesis_pip, 'install', '--upgrade', 'pip', '--proxy', PROXY], capture_output=True, text=True)
    print(f'Created venv: {thesis_env}')

print(f'\nthesis_env path   : {thesis_env}')
print(f'thesis_env python : {thesis_python}')

## c03 — Search compiler & load CUDA modules
Locates system `nvcc` and modules to check compiler compatibility.

In [ ]:
# ── Search nvcc & load CUDA modules ───────────────────────────────────────────
import subprocess
import os

search_script = r'''
which nvcc 2>/dev/null && exit 0
for init in /etc/profile /etc/profile.d/modules.sh \
            /usr/share/lmod/lmod/init/bash /usr/share/lmod/lmod/init/sh; do
    [ -f "$init" ] && source "$init" 2>/dev/null
done
for mod in cuda/11.7 cuda/11.8 cuda/11.2 cuda/11 cuda CUDA/11.7 CUDA/11.8 CUDA cuda-11.7 cuda-11.8 cuda-11 cuda/12.6 cuda/12.1 cuda/12 cuda CUDA/12.6 CUDA/12.1 cuda-12.6 cuda-12-1 cuda-12; do
    module load "$mod" 2>/dev/null
    nv=$(which nvcc 2>/dev/null); [ -n "$nv" ] && echo "$nv" && exit 0
done
for base in /fs /work /gpfs /scratch /software /apps /appl /tools /opt/software /usr/local; do
    [ -d "$base" ] || continue
    result=$(find "$base" -name nvcc -type f -maxdepth 8 2>/dev/null | head -1)
    [ -n "$result" ] && echo "$result" && exit 0
done
'''

r_search = subprocess.run(['bash', '-c', search_script], capture_output=True, text=True, timeout=90)
nvcc_path = r_search.stdout.strip()
print(f'nvcc found: {repr(nvcc_path)}')
if not nvcc_path or not os.path.isfile(nvcc_path):
    raise RuntimeError('nvcc not found — check CUDA module availability on server')

cuda_home = os.path.dirname(os.path.dirname(nvcc_path))
os.environ['CUDA_HOME'] = cuda_home

print(f'CUDA_HOME : {cuda_home}')

## c04 — Auto-Install PyTorch
Dynamically detects the CUDA compiler version and installs the matching PyTorch runtime.

In [ ]:
# ── Auto-Install PyTorch matching CUDA compiler ────────────────────────────────
import subprocess
import os
import sys

# Get CUDA version from nvcc --version
r_ver = subprocess.run([nvcc_path, '--version'], capture_output=True, text=True)
version_out = r_ver.stdout
print(version_out)

# Get thesis_env Python version dynamically
r_py = subprocess.run([thesis_python, '--version'], capture_output=True, text=True)
thesis_py_ver_str = r_py.stdout.strip()
print(f"thesis_env Python version: {thesis_py_ver_str}")

# Extract major.minor, e.g. "3.10", "3.11"
py_ver = "3.10"
if "3.11" in thesis_py_ver_str:
    py_ver = "3.11"
elif "3.12" in thesis_py_ver_str:
    py_ver = "3.12"

if 'release 11.' in version_out:
    if py_ver == '3.11':
        print('CUDA 11 + Python 3.11 detected! Installing PyTorch 2.0.1 + cu117.')
        torch_spec = 'torch==2.0.1+cu117'
        torchvision_spec = 'torchvision==0.15.2+cu117'
        index_url = 'https://download.pytorch.org/whl/cu117'
    elif py_ver == '3.12':
        print('CUDA 11 + Python 3.12 detected! Installing PyTorch 2.2.2 + cu118.')
        torch_spec = 'torch==2.2.2+cu118'
        torchvision_spec = 'torchvision==0.17.2+cu118'
        index_url = 'https://download.pytorch.org/whl/cu118'
    else:
        print('CUDA 11 detected! Installing PyTorch 1.13.1 + cu117 (Kaggle thesis baseline).')
        torch_spec = 'torch==1.13.1+cu117'
        torchvision_spec = 'torchvision==0.14.1+cu117'
        index_url = 'https://download.pytorch.org/whl/cu117'
elif 'release 12.6' in version_out:
    print('CUDA 12.6 detected! Installing PyTorch 2.6.0 + cu126.')
    torch_spec = 'torch==2.6.0+cu126'
    torchvision_spec = 'torchvision==0.21.0+cu126'
    index_url = 'https://download.pytorch.org/whl/cu126'
elif 'release 12.' in version_out:
    if py_ver == '3.12':
        print('CUDA 12.x + Python 3.12 detected! Installing PyTorch 2.2.2 + cu121.')
        torch_spec = 'torch==2.2.2+cu121'
        torchvision_spec = 'torchvision==0.17.2+cu121'
        index_url = 'https://download.pytorch.org/whl/cu121'
    else:
        print('CUDA 12.x detected! Installing PyTorch 2.1.2 + cu121.')
        torch_spec = 'torch==2.1.2+cu121'
        torchvision_spec = 'torchvision==0.16.2+cu121'
        index_url = 'https://download.pytorch.org/whl/cu121'
else:
    print('Unknown CUDA version. Defaulting to PyTorch 1.13.1+cu117.')
    torch_spec = 'torch==1.13.1+cu117'
    torchvision_spec = 'torchvision==0.14.1+cu117'
    index_url = 'https://download.pytorch.org/whl/cu117'

r_install = subprocess.run([
    thesis_pip, 'install', torch_spec, torchvision_spec,
    '--index-url', index_url, '--proxy', PROXY
], capture_output=True, text=True)

print('PyTorch Install Status:', 'OK' if r_install.returncode == 0 else r_install.stderr[-1000:])

## c05 — Install Base Dependencies
Installs base packages and checks PyTorch's CUDA backend validation.

In [ ]:
# ── Install Base Dependencies ─────────────────────────────────────────────────
import subprocess

# Pin numpy<2 to avoid compatibility breakages with PyTorch 1.x / 3DGS extensions
base_deps = [
    'wheel', 'setuptools', 'ipykernel', 'tensorboard', 'pandas', 
    'plyfile', 'tqdm', 'ninja', 'scikit-image', 'imageio', 'scipy', 'websockets', 'numpy<2', 'huggingface_hub'
]

print('Installing base pip packages...')
r = subprocess.run([thesis_pip, 'install'] + base_deps + ['--proxy', PROXY], capture_output=True, text=True)
print('Base dependencies output:', 'OK' if r.returncode == 0 else r.stderr[-1000:])

# Re-run a check of the python path torch CUDA availability
verify_torch = """
import torch
print('Torch version:', torch.__version__)
print('CUDA back-end:', torch.version.cuda)
print('GPU Available:', torch.cuda.is_available())
"""
r_v = subprocess.run([thesis_python, '-c', verify_torch], capture_output=True, text=True)
print('\nTorch diagnostics inside venv:\n', r_v.stdout if r_v.returncode == 0 else r_v.stderr)

## c06 — Build and Install Custom CUDA Submodules
Compiles the custom submodules (`diff-gaussian-rasterization_fastgs`, `simple-knn`, `fused-ssim`) into wheels and installs them directly.

In [ ]:
# ── Build and Install Custom CUDA Submodules ─────────────────────────────────
import subprocess
import os
import sys
import glob

# Ensure compiler and library paths are exported
sys_cc  = subprocess.run(['which', 'gcc'], capture_output=True, text=True).stdout.strip() or '/usr/bin/gcc'
sys_cxx = subprocess.run(['which', 'g++'], capture_output=True, text=True).stdout.strip() or '/usr/bin/g++'

# Retrieve host compiler and library paths
r_find_rt = subprocess.run(['find', thesis_env, '-name', 'libcudart*', '-type', 'f'],
                           capture_output=True, text=True, timeout=30)
rt_libs = [p.strip() for p in r_find_rt.stdout.strip().split('\n') if p.strip()]
cuda_rt_lib = os.path.dirname(rt_libs[0]) if rt_libs else ''
ld_path = f'{cuda_home}/lib64:{cuda_rt_lib}' if cuda_rt_lib else f'{cuda_home}/lib64'

# Try to get GPU compute capability dynamically via torch
r_cc = subprocess.run([thesis_python, '-c', "import torch; print(f'{torch.cuda.get_device_capability(0)[0]}.{torch.cuda.get_device_capability(0)[1]}')"], capture_output=True, text=True)
cc = r_cc.stdout.strip() if r_cc.returncode == 0 else "8.0"
print(f'CUDA_HOME : {cuda_home}')
print(f'CC/CXX    : {sys_cc} / {sys_cxx}')
print(f'GPU Arch   : Compute Capability {cc}')

# Check for GLM third party header download just in case recursive clone was incomplete
glm_dir = os.path.join(SUBMODULES_DIR, 'diff-gaussian-rasterization_fastgs', 'third_party', 'glm')
if not (os.path.isdir(glm_dir) and os.listdir(glm_dir)):
    print('GLM not found in third_party. Downloading it...')
    os.makedirs(os.path.dirname(glm_dir), exist_ok=True)
    git_proxy = ['-c', f'http.proxy={PROXY}', '-c', f'https.proxy={PROXY}']
    r = subprocess.run(['git'] + git_proxy + ['clone', 'https://github.com/g-truc/glm.git', glm_dir, '--depth=1'],
                       capture_output=True, text=True, timeout=300)
    print('GLM cloned OK' if r.returncode == 0 else f'GLM clone failed: {r.stderr}')

def build_ext(name, src_dir):
    print(f'\n=== Building {name} ===')
    if not os.path.isdir(src_dir):
        print(f'ERROR: source dir not found: {src_dir}')
        return False
    build_cmd = (
        f'cd {src_dir} && rm -rf build dist *.egg-info && '
        f'CUDA_HOME={cuda_home} PATH={cuda_home}/bin:$PATH '
        f'LD_LIBRARY_PATH={ld_path}:$LD_LIBRARY_PATH '
        f'TORCH_CUDA_ARCH_LIST={cc} CC={sys_cc} CXX={sys_cxx} '
        f'{thesis_python} setup.py bdist_wheel 2>&1'
    )
    r = subprocess.run(build_cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print('BUILD STDERR:', r.stderr[-1000:])
        return False
    wheels = glob.glob(f'{src_dir}/dist/*.whl')
    if not wheels:
        print(f'ERROR: No wheel in {src_dir}/dist/')
        return False
    ri = subprocess.run([thesis_pip, 'install', wheels[0], '--no-deps', '--force-reinstall'],
                        capture_output=True, text=True)
    if ri.returncode != 0:
        print('INSTALL STDERR:', ri.stderr[-500:])
        return False
    print(f'{name}: OK')
    return True

exts = [
    ('diff-gaussian-rasterization_fastgs', os.path.join(SUBMODULES_DIR, 'diff-gaussian-rasterization_fastgs')),
    ('simple-knn', os.path.join(SUBMODULES_DIR, 'simple-knn')),
    ('fused-ssim', os.path.join(SUBMODULES_DIR, 'fused-ssim'))
]

results = {name: build_ext(name, src) for name, src in exts}
print('\n=== Build Summary ===')
for name, ok in results.items():
    print(f'  {name:40s}: {"OK" if ok else "FAILED"}')

## c07 — Register Jupyter Kernel
Registers the python environment as an available kernel named `thesis_env`.

In [ ]:
# ── Register Jupyter Kernel ───────────────────────────────────────────────────
import subprocess

subprocess.run([thesis_pip, 'install', 'ipykernel', '--proxy', PROXY], capture_output=True, text=True)
r_k = subprocess.run([
    thesis_python, '-m', 'ipykernel', 'install',
    '--user', '--name', ENV_NAME,
    '--display-name', f'Python ({ENV_NAME})'
], capture_output=True, text=True)
print('Kernel Register Output:', 'OK' if r_k.returncode == 0 else r_k.stderr[-300:])

## c08 — Verify Imports
Imports verification check for PyTorch CUDA support and Spec-FastGS modules.

In [ ]:
# ── Verify Imports ───────────────────────────────────────────────────────────
import subprocess

verify_script = """
import torch
import diff_gaussian_rasterization_fastgs
import simple_knn
import fused_ssim
import plyfile
import websockets
import tqdm
import huggingface_hub
print('✅ Torch CUDA Available:', torch.cuda.is_available())
print('✅ Rasterizer fastgs   : OK')
print('✅ simple-knn          : OK')
print('✅ fused-ssim          : OK')
print('✅ huggingface_hub     : OK')
print('🎉 All systems functional and ready for execution!')
"""
r_v = subprocess.run([thesis_python, '-c', verify_script], capture_output=True, text=True)
print(r_v.stdout if r_v.returncode == 0 else f'Verification Failed:\n{r_v.stderr}')